::: {.sb-downloads}
[Download this notebook](../downloads/solutions-feature-engineering.ipynb){download="solutions-feature-engineering.ipynb"}  
Data: [california_cities.csv](../downloads/data/california_cities.csv){download="california_cities.csv"} · [california_housing_market.csv](../downloads/data/california_housing_market.csv){download="california_housing_market.csv"}

Put the data files in a folder named `data` next to the notebook.
:::

::: {.sb-lesson-links}
[← Back to the exercises](../exercises/feature-engineering.ipynb)
:::

::: {.callout-tip}
Try the exercises on your own first: you learn much more by searching than by reading a solution.
:::

In the previous exercise, we concluded that it was worth including more variables in the model. But is this set of variables **the best** we could have chosen? In this exercise, we'll go further by applying two classic methods:

* Feature engineering consists in creating more variables from the original dataset
* Feature selection selects the best set of features among all the available variables

## The dataset {#ex-the-dataset}

1. Load the California Housing dataset again and remove the outliers:

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import  OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) # to avoid deprecation warnings

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
# colour palette of the course
pio.templates["course"] = go.layout.Template(
    layout_colorway=["#3B47C4", "#E4572E", "#2E7D4F", "#F2C14E", "#8E7CC3", "#4BA3C3", "#111111", "#999999"]
)
pio.templates.default = "plotly_white+course"


In [ ]:
dataset = pd.read_csv("../data/california_housing_market.csv")
dataset = dataset.rename(columns={"MedHouseVal": "Price"})  # median house value, in $100,000
dataset.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,Price
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [ ]:
# Remove outliers
mask = (dataset['AveRooms'] < 10) & (dataset['AveBedrms'] < 10) & (dataset['Population'] < 15000) & \
    (dataset['AveOccup'] < 10) & (dataset['Price'] < 5)
dataset = dataset.loc[mask,:]

In [ ]:
# Basic stats
print("Number of rows : {}".format(dataset.shape[0]))
print()

print("Display of dataset: ")
display(dataset.head())
print()

print("Basics statistics: ")
data_desc = dataset.describe(include='all')
display(data_desc)
print()

print("Percentage of missing values: ")
display(100*dataset.isnull().sum()/dataset.shape[0])

Number of rows : 19398

Display of dataset: 


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,Price
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422



Basics statistics: 


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,Price
count,19398.000000,19398.000000,19398.000000,19398.000000,19398.000000,19398.000000,19398.000000,19398.000000,19398.000000
mean,3.674497,28.496907,5.210648,1.066038,1442.172080,2.944640,35.637872,-119.567484,1.924128
std,1.563397,12.477953,1.168098,0.128846,1077.498768,0.766194,2.142960,2.004793,0.971784
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.750000,32.540000,-124.350000,0.149990
25%,2.525900,18.000000,4.407329,1.005413,805.000000,2.450413,33.930000,-121.770000,1.167000
50%,3.447800,29.000000,5.170038,1.047619,1185.500000,2.842105,34.260000,-118.490000,1.741000
75%,4.583175,37.000000,5.944617,1.096884,1752.000000,3.308127,37.720000,-118.000000,2.485000
max,15.000100,52.000000,9.979167,3.411111,13251.000000,9.954545,41.950000,-114.550000,4.991000



Percentage of missing values: 


MedInc        0.0
HouseAge      0.0
AveRooms      0.0
AveBedrms     0.0
Population    0.0
AveOccup      0.0
Latitude      0.0
Longitude     0.0
Price         0.0
dtype: float64

2. Separate the target from the features

In [ ]:
# Separate target variable Y from features X
print("Separating labels from features...")
target_variable = "Price"

X = dataset.drop(target_variable, axis = 1)
Y = dataset.loc[:,target_variable]

print("...Done.")
print()

print('Y : ')
print(Y.head())
print()
print('X :')
X.head()

Separating labels from features...
...Done.

Y : 
0    4.526
1    3.585
2    3.521
3    3.413
4    3.422
Name: Price, dtype: float64

X :


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


## From linear to non-linear regression {#ex-from-linear-to-non-linear-regression}
An easy way to implement a non-linear regression is to create, by hand, new columns containing non-linear functions of the features.

3. For each explanatory variable, create 5 new columns in $X$ containing the following functions:
* $\textrm{X}^2$
* $\textrm{X}^3$
* $\textrm{X}^4$
* $\frac{1}{\textrm{X}}$
* $\frac{1}{\textrm{X}^2}$

In [ ]:
features_list = X.columns
for c in features_list:
    X.loc[:, c + '_2'] = X[c]**2
    X.loc[:, c + '_3'] = X[c]**3
    X.loc[:, c + '_4'] = X[c]**4
    X.loc[:, c + '_inverse'] = 1/X[c]
    X.loc[:, c + '_inverse2'] = 1/(X[c]**2)
X.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedInc_2,MedInc_3,...,Latitude_2,Latitude_3,Latitude_4,Latitude_inverse,Latitude_inverse2,Longitude_2,Longitude_3,Longitude_4,Longitude_inverse,Longitude_inverse2
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,69.308955,577.010912,...,1434.8944,54353.799872,2.058922e+06,0.026399,0.000697,14940.1729,-1.826137e+06,2.232088e+08,-0.008181,0.000067
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,68.913242,572.076387,...,1433.3796,54267.751656,2.054577e+06,0.026413,0.000698,14937.7284,-1.825689e+06,2.231357e+08,-0.008182,0.000067
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,52.669855,382.246204,...,1432.6225,54224.761625,2.052407e+06,0.026420,0.000698,14942.6176,-1.826586e+06,2.232818e+08,-0.008181,0.000067
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,31.844578,179.702136,...,1432.6225,54224.761625,2.052407e+06,0.026420,0.000698,14945.0625,-1.827034e+06,2.233549e+08,-0.008180,0.000067
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,14.793254,56.897815,...,1432.6225,54224.761625,2.052407e+06,0.026420,0.000698,14945.0625,-1.827034e+06,2.233549e+08,-0.008180,0.000067


4. Split your dataset into train (80%) and test (20%)

In [ ]:
# Divide dataset Train set & Test set 
print("Dividing into train and test sets...")
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)
print("...Done.")
print()

Dividing into train and test sets...
...Done.



5. Apply the same preprocessing as in the previous exercise

In [ ]:
# Preprocessing
print("Preprocessing X_train...")
print(X_train.head())
print()
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
print("...Done!")
print(X_train[0:5,:]) # X_train is now a numpy array

Preprocessing X_train...
       MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
3235   2.3889       6.0  6.316614   1.294671       992.0  3.109718     36.09   
13981  3.4912       7.0  8.355308   1.554795      2933.0  2.511130     34.85   
9219   1.9464      36.0  4.975510   1.053061       639.0  2.608163     37.12   
10851  3.1667      22.0  3.803838   1.000000      1952.0  2.081023     33.66   
8888   4.2520      31.0  3.978296   1.039389      1985.0  1.595659     34.03   

       Longitude   MedInc_2   MedInc_3  ...  Latitude_2    Latitude_3  \
3235     -119.57   5.706843  13.633078  ...   1302.4881  47006.795529   
13981    -117.46  12.188477  42.552412  ...   1214.5225  42326.109125   
9219     -120.27   3.788473   7.373884  ...   1377.8944  51147.440128   
10851    -117.90  10.027989  31.755632  ...   1132.9956  38136.631896   
8888     -118.49  18.079504  76.874051  ...   1158.0409  39408.131827   

         Latitude_4  Latitude_inverse  Latitude_inverse

In [ ]:
print("Preprocessing X_test...")
print(X_test.head())
print()
X_test = scaler.transform(X_test) # don't fit again !
print("...Done!")
print(X_test[0:5,:]) # X_train is now a numpy array

Preprocessing X_test...
       MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
17333  5.2990      12.0  7.214932   1.047511      1200.0  2.714932     34.91   
1012   2.6667      44.0  4.541284   1.027523       277.0  2.541284     37.68   
5124   1.5521      30.0  3.850679   1.002262      1966.0  4.447964     33.99   
1845   6.3538      49.0  6.293886   1.017751      1148.0  2.264300     37.90   
4035   3.2154      20.0  4.133444   1.060181      7450.0  1.772122     34.17   

       Longitude   MedInc_2    MedInc_3  ...  Latitude_2    Latitude_3  \
17333    -120.44  28.079401  148.792746  ...   1218.7081  42545.099771   
1012     -121.77   7.111289   18.963674  ...   1419.7824  53497.400832   
5124     -118.26   2.409014    3.739031  ...   1155.3201  39269.330199   
1845     -122.28  40.370774  256.507827  ...   1436.4100  54439.939000   
4035     -118.52  10.338797   33.243368  ...   1167.5889  39896.512713   

         Latitude_4  Latitude_inverse  Latitude_in

6. Train a model including all these features. Do you get better performance than before?

In [ ]:
# Train model
print("Train model...")
regressor = LinearRegression()
regressor.fit(X_train, Y_train)
print("...Done.")

Train model...
...Done.


In [ ]:
# Print R^2 scores
print("R2 score on training set : ", regressor.score(X_train, Y_train))
print("R2 score on test set : ", regressor.score(X_test, Y_test))

R2 score on training set :  0.6729622311012318
R2 score on test set :  0.6755800683925868


## Forward selection {#ex-forward-selection}
This feature engineering trick improved the model's score (test $R^2$ from 0.64 to 0.68)! But now, the model is a lot more complex, as it uses 48 input features. Do we really need all these features? Let's implement the forward selection method described in the lecture. 

Fortunately, recent versions of sklearn provide a class that implements forward selection, so we don't need to code the algorithm by hand 🥳

7. Have a look at the documentation of [SequentialFeatureSelector](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SequentialFeatureSelector.html) and try to understand the following lines of code:

In [ ]:
from sklearn.feature_selection import  SequentialFeatureSelector
feature_selector =  SequentialFeatureSelector(regressor, n_features_to_select = 20)
feature_selector.fit(X_train, Y_train)
features_list = X.columns
best_features = features_list[feature_selector.support_]
print("According to the forward selection algorithm, the following features should be kept: ")
print(best_features.to_list())

According to the forward selection algorithm, the following features should be kept: 
['MedInc', 'HouseAge', 'Population', 'MedInc_3', 'MedInc_4', 'MedInc_inverse2', 'AveRooms_3', 'AveRooms_inverse', 'AveRooms_inverse2', 'AveBedrms_inverse', 'Population_2', 'Population_inverse2', 'AveOccup_3', 'AveOccup_inverse', 'AveOccup_inverse2', 'Latitude_3', 'Latitude_4', 'Latitude_inverse', 'Latitude_inverse2', 'Longitude_inverse']


8. Create a DataFrame X_best containing only the best set of features, train a model with these features only, and evaluate its performance

In [ ]:
X_best = X.loc[:, best_features]

# Divide dataset Train set & Test set 
print("Dividing into train and test sets...")
X_train, X_test, Y_train, Y_test = train_test_split(X_best, Y, test_size=0.2, random_state=0)
print("...Done.")
print()

# Preprocessing
print("Preprocessing X_train...")
print(X_train.head())
print()
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
print("...Done!")
print(X_train[0:5,:]) # X_train is now a numpy array

print("Preprocessing X_test...")
print(X_test.head())
print()
X_test = scaler.transform(X_test) # don't fit again !
print("...Done!")
print(X_test[0:5,:]) # X_train is now a numpy array

# Train model
print("Train model...")
regressor = LinearRegression()
regressor.fit(X_train, Y_train)
print("...Done.")

# Print R^2 scores
print("R2 score on training set : ", regressor.score(X_train, Y_train))
print("R2 score on test set : ", regressor.score(X_test, Y_test))

Dividing into train and test sets...
...Done.

Preprocessing X_train...
       MedInc  HouseAge  Population   MedInc_3    MedInc_4  MedInc_inverse2  \
3235   2.3889       6.0       992.0  13.633078   32.568059         0.175228   
13981  3.4912       7.0      2933.0  42.552412  148.558982         0.082045   
9219   1.9464      36.0       639.0   7.373884   14.352527         0.263959   
10851  3.1667      22.0      1952.0  31.755632  100.560561         0.099721   
8888   4.2520      31.0      1985.0  76.874051  326.868465         0.055311   

       AveRooms_3  AveRooms_inverse  AveRooms_inverse2  AveBedrms_inverse  \
3235   252.030501          0.158313           0.025063           0.772397   
13981  583.293888          0.119684           0.014324           0.643172   
9219   123.172247          0.200984           0.040395           0.949612   
10851   55.038428          0.262892           0.069112           1.000000   
8888    62.963842          0.251364           0.063184           0.9

**Thanks to forward selection, we kept only 20 features and got almost the same performance (test $R^2$ 0.67) as with the 48 features (0.68): a simpler model for the same quality.** 🤓

## Advanced feature engineering: from coordinates to geography {#ex-advanced-feature-engineering-from-coordinates-to-geography}
Until now, latitude and longitude went into the model as they are. A linear model can only learn something like "the further north, the cheaper", while prices actually depend on **where** a district is: close to a big city, prices are much higher.

Let's turn the GPS coordinates into geographic features. The file `../data/california_cities.csv` gives the coordinates of ten big cities of California, and the function below computes the distance in km between two points on Earth (the [haversine formula](https://en.wikipedia.org/wiki/Haversine_formula)).

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    """Distance in km between two points (or arrays of points) given by their latitude and longitude."""
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

cities = pd.read_csv("../data/california_cities.csv")
cities

,city,latitude,longitude
0,Los Angeles,34.0522,-118.2437
1,San Diego,32.7157,-117.1611
2,San Jose,37.3382,-121.8863
3,San Francisco,37.7749,-122.4194
4,Fresno,36.7378,-119.7871
5,Sacramento,38.5816,-121.4944
6,Oakland,37.8044,-122.2712
7,Bakersfield,35.3733,-119.0187
8,Riverside,33.9806,-117.3755
9,Santa Barbara,34.4208,-119.6982


9. Start from the original features (`dataset` without the target). For each district, compute its distance to each city, then add two columns:
* `nearest_city`: the name of the closest city
* `distance_to_city`: the distance to this city, in km

::: {.callout-tip collapse="true" title="Show hint"}
Build a DataFrame with one column per city, then use `.idxmin(axis=1)` and `.min(axis=1)`.
:::

In [ ]:
X_geo = dataset.drop(target_variable, axis=1)

distances = pd.DataFrame({
    city.city: haversine(X_geo["Latitude"], X_geo["Longitude"], city.latitude, city.longitude)
    for city in cities.itertuples()
}, index=X_geo.index)

X_geo["nearest_city"] = distances.idxmin(axis=1)
X_geo["distance_to_city"] = distances.min(axis=1)
X_geo[["Latitude", "Longitude", "nearest_city", "distance_to_city"]].head()

,Latitude,Longitude,nearest_city,distance_to_city
0,37.88,-122.23,Oakland,9.151779
1,37.86,-122.22,Oakland,7.644694
2,37.85,-122.24,Oakland,5.763582
3,37.85,-122.25,Oakland,5.401555
4,37.85,-122.25,Oakland,5.401555


10. Draw a map of the districts (longitude on x, latitude on y), colored by `nearest_city`. Then a second one colored by the price. What do you see?

In [ ]:
fig = px.scatter(X_geo, x="Longitude", y="Latitude", color="nearest_city", opacity=0.5,
                 title="Districts by nearest big city", height=650)
fig.update_yaxes(scaleanchor="x")
fig.show()

fig = px.scatter(dataset, x="Longitude", y="Latitude", color="Price", opacity=0.5,
                 color_continuous_scale="Viridis", title="Median house value", height=650)
fig.update_yaxes(scaleanchor="x")
fig.show()

**The most expensive districts are on the coast, around San Francisco, Los Angeles and San Diego. Inland, prices drop as we move away from the cities.**

11. `nearest_city` is categorical: it needs a one-hot encoding, while the numerical features need a standardization. Build a `ColumnTransformer`, train a linear regression on `X_geo`, and compare its $R^2$ with the multivariate model of the previous exercise (8 features, $R^2 \approx 0.64$ on the test set) and with the 48 polynomial features above.

In [ ]:
categorical_features = ["nearest_city"]
numeric_features = [c for c in X_geo.columns if c not in categorical_features]

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(drop="first"), categorical_features),
])

X_train, X_test, Y_train, Y_test = train_test_split(X_geo, Y, test_size=0.2, random_state=0)
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)  # don't fit again!

regressor = LinearRegression()
regressor.fit(X_train, Y_train)
print("R2 score on training set : ", regressor.score(X_train, Y_train))
print("R2 score on test set : ", regressor.score(X_test, Y_test))

R2 score on training set :  0.6673475586161636
R2 score on test set :  0.6752570131023358


12. Bonus: combine everything, the polynomial features of question 3 **and** the two geographic features. Is it better?

In [ ]:
X_all = pd.concat([X, X_geo[["nearest_city", "distance_to_city"]]], axis=1)
numeric_features = [c for c in X_all.columns if c not in categorical_features]
preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(drop="first"), categorical_features),
])

X_train, X_test, Y_train, Y_test = train_test_split(X_all, Y, test_size=0.2, random_state=0)
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

regressor = LinearRegression()
regressor.fit(X_train, Y_train)
print("Number of features: ", X_train.shape[1])
print("R2 score on training set : ", regressor.score(X_train, Y_train))
print("R2 score on test set : ", regressor.score(X_test, Y_test))

Number of features:  58
R2 score on training set :  0.7093787160345151
R2 score on test set :  0.713562845595513


## Conclusion {#ex-conclusion}

* **Two well-chosen features** (the nearest city and the distance to it) improve the model as much as the 40 polynomial features: feature engineering works best when it brings **knowledge about the problem**, not just more columns.
* Combining both gives the best model of this exercise.
* Each new feature makes the model more complex. With many features and few examples, a model starts to learn the noise of the training set: this is **overfitting**, the subject of the next section 🤓